In [ ]:
# 1. Install & Import Required Libraries
!pip install scikit-learn pandas numpy

import pandas as pd
import numpy as np
import urllib.request
import zipfile
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

In [ ]:
# 2. Load the SMS Spam Collection Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
urllib.request.urlretrieve(url, "smsspamcollection.zip")
with zipfile.ZipFile("smsspamcollection.zip", "r") as zip_ref:
    zip_ref.extractall("sms_data")

df = pd.read_csv("sms_data/SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
# 3. Split Data into Train/Test Sets
X = df['message']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 4. Vectorize Text & Train the Model
vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

In [ ]:
# 5. Evaluate Baseline Performance
y_pred = model.predict(X_test_vec)
print("--- Baseline Model Performance ---")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision (spam):", precision_score(y_test, y_pred, pos_label="spam"))
print("Recall (spam):", recall_score(y_test, y_pred, pos_label="spam"))

--- Baseline Model Performance ---
Accuracy: 0.9668161434977578
Precision (spam): 1.0
Recall (spam): 0.7516778523489933


In [ ]:
# 6. Test Adversarial Messages & Build Results Table
test_messages = [
    "Cl@im your fr33 g1ft card now",          # Trick 1: Spelling/obfuscation
    "Not free, just confirming lunch at 1pm",# Trick 2: Negation
    "URGENT! You have won a free vacation. Click here!", # Trick 3: Classic spam words
    "Hey, are we still meeting for our project review tomorrow?", # Trick 4: Normal email
    "Congratulations, cash prize waiting for you!" # Trick 5: Direct spam
]

test_labels = ["spam", "ham", "spam", "ham", "spam"]

# Run predictions
test_vecs = vec.transform(test_messages)
predictions = model.predict(test_vecs)

# Document the explanations of success/failure
failure_explanations = [
    "Fooled: 'Cl@im', 'fr33' and 'g1ft' tokenize into unseen tokens ('cl', 'im', 'fr33', 'g1ft') with zero spam evidence, so ordinary words like 'your', 'card' and 'now' decide the vote for ham.",
    "Right answer, wrong reason: bag-of-words cannot parse the negation 'Not free'; everyday words ('just', 'lunch', 'pm') and the strong ham prior simply outweighed the trigger word 'free'.",
    "Fooled: TF-IDF length normalization dilutes the trigger words across 8 tokens, and words like 'urgent', 'you' and 'have' also occur in personal texts, so the ~87% ham prior tips it to ham.",
    "Correct: plain conversational wording with no prize or offer trigger words looks exactly like the ham messages the model learned from.",
    "Caught: 'congratulations' and 'cash prize' are strong, correctly spelled spam triggers that dominate the short message's TF-IDF weights."
]

# Construct and display final results table
results_data = []
for msg, pred, true_lbl, explanation in zip(test_messages, predictions, test_labels, failure_explanations):
    results_data.append({
        "My Message": msg,
        "Model Said": pred,
        "Should Be": true_lbl,
        "Why It Succeeded / Failed": explanation
    })

results_df = pd.DataFrame(results_data)
display(results_df)

,My Message,Model Said,Should Be,Why It Succeeded / Failed
0,Cl@im your fr33 g1ft card now,ham,spam,"Fooled: 'Cl@im', 'fr33' and 'g1ft' tokenize in..."
1,"Not free, just confirming lunch at 1pm",ham,ham,"Right answer, wrong reason: bag-of-words canno..."
2,URGENT! You have won a free vacation. Click here!,ham,spam,Fooled: TF-IDF length normalization dilutes th...
3,"Hey, are we still meeting for our project revi...",ham,ham,Correct: plain conversational wording with no ...
4,"Congratulations, cash prize waiting for you!",spam,spam,Caught: 'congratulations' and 'cash prize' are...


## Reflection
Through this project, I learned that a classical Naive Bayes spam filter can look excellent on paper (96.7% accuracy, perfect precision) and still be fooled by cheap tricks like character substitutions or a longer, wordier message that dilutes the trigger words. Because TF-IDF treats every word as an independent token, the model has no sense of context or obfuscation - it only catches spam phrased like the spam it was trained on, which is exactly why adversarial testing and modern language-model filters matter.

In [ ]:
# 1. Import required modules
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
import urllib.request
import zipfile
import os

# 2. Load the SMS Spam Collection dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
urllib.request.urlretrieve(url, "smsspamcollection.zip")
with zipfile.ZipFile("smsspamcollection.zip", "r") as zip_ref:
    zip_ref.extractall("sms_data")

df = pd.read_csv("sms_data/SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])

# 3. Train/Test Split
X = df['message']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Fit Vectorizer and Train Model
vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train)
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# 5. Define Adversarial Messages and Predict
test_messages = [
    "Cl@im your fr33 g1ft card now",          # Trick 1: Spelling/obfuscation
    "Not free, just confirming lunch at 1pm",# Trick 2: Negation
    "URGENT! You have won a free vacation. Click here!", # Trick 3: Classic spam words
    "Hey, are we still meeting for our project review tomorrow?", # Trick 4: Normal email
    "Congratulations, cash prize waiting for you!" # Trick 5: Direct spam
]

test_vecs = vec.transform(test_messages)
predictions = model.predict(test_vecs)

# 6. Analyze Results
test_labels = ["spam", "ham", "spam", "ham", "spam"]
results = []
for msg, pred, true_label in zip(test_messages, predictions, test_labels):
    fooled = "Yes" if pred != true_label else "No"
    results.append({
        "Message": msg,
        "Expected": true_label,
        "Predicted": pred,
        "Fooled Filter?": fooled
    })

results_df = pd.DataFrame(results)
display(results_df)

,Message,Expected,Predicted,Fooled Filter?
0,Cl@im your fr33 g1ft card now,spam,ham,Yes
1,"Not free, just confirming lunch at 1pm",ham,ham,No
2,URGENT! You have won a free vacation. Click here!,spam,ham,Yes
3,"Hey, are we still meeting for our project revi...",ham,ham,No
4,"Congratulations, cash prize waiting for you!",spam,spam,No
